In [1]:
from pathlib import Path

import duckdb
from config.mapping_spreadsheets import SPREADSHEET_MAPPING

In [2]:
SPREADSHEET_MAPPING.sheets[8:]

[SheetSource(spreadsheet_id='1ShmRNwBD6AMQtP0BiTuXxhvOg2zXyUEPCoXVQoIhRhE', easybill_spreadsheet_name='1.Clientlist', medisoft_spreadsheet_name='2.medisoft_Viersen_firms', city='Viersen', description='Easybill and medisoft client lists for Viersen.', url='https://docs.google.com/spreadsheets/d/1ShmRNwBD6AMQtP0BiTuXxhvOg2zXyUEPCoXVQoIhRhE/edit'),
 SheetSource(spreadsheet_id='1DynAAKo8sHkGx6TrfSQDlaqyQ5N8jcThXFWGh7k1IlQ', easybill_spreadsheet_name='1.Clientlist', medisoft_spreadsheet_name='2.medisoft_berlin_firms', city='Berlin', description='Easybill and medisoft client lists for Berlin.', url='https://docs.google.com/spreadsheets/d/1DynAAKo8sHkGx6TrfSQDlaqyQ5N8jcThXFWGh7k1IlQ/edit')]

In [3]:
def _sql_literal(value: str) -> str:
    return value.replace("'", "''")


def _sql_identifier(value: str) -> str:
    return '"' + value.replace('"', '""') + '"'

In [4]:
# Create a table for each city spreadsheet (easybill tab)
credentials_path = Path("config/gsheet-duckdb-e1ebeef322e4.json").resolve()

con = duckdb.connect()
con.execute("INSTALL gsheets FROM community;")
con.execute("LOAD gsheets;")
con.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{_sql_literal(str(credentials_path))}'
);
"""
)

In [5]:
for source in SPREADSHEET_MAPPING.sheets:
    table_name = _sql_identifier(source.city)
    spreadsheet_id = _sql_literal(source.spreadsheet_id)
    sheet_name = _sql_literal(source.easybill_spreadsheet_name)

    con.execute(
        f"""
CREATE OR REPLACE TABLE {table_name} AS
SELECT *
FROM read_gsheet(
    '{spreadsheet_id}',
    sheet='{sheet_name}',
    all_varchar=true
);
"""
    )

con.sql("SHOW TABLES").df()

,name
0,Berlin
1,Düsseldorf
2,Frankfurt
3,Hamburg
4,Kiel
5,Köln
6,München
7,Rostock
8,Stuttgart
9,Viersen


In [7]:
con.sql(
    """
SELECT 
    easybill_kundennummer,
    unnest(COALESCE(string_split(medisoft_ids, '\n'), [NULL])) AS medisoft_id
FROM berlin;
    """)

┌───────────────────────┬───────────────┐
│ easybill_kundennummer │  medisoft_id  │
│        varchar        │    varchar    │
├───────────────────────┼───────────────┤
│ 130002124             │ 00_A1W00VHI68 │
│ 130002124             │ 00_8JJ00VTGCE │
│ 130000200             │ NULL          │
│ 130000437             │ NULL          │
│ 130002066             │ 00_A0S00RIHE7 │
│ 111000009             │ 00_8PG00L8MCI │
│ 113010031             │ NULL          │
│ 130000392             │ 00_9NW00RRFMP │
│ 127000001             │ NULL          │
│ 127000000             │ NULL          │
│     ·                 │  ·            │
│     ·                 │  ·            │
│     ·                 │  ·            │
│ 130002257             │ NULL          │
│ 130002263             │ NULL          │
│ 130002223             │ NULL          │
│ 130002269             │ NULL          │
│ 130002270             │ NULL          │
│ 130001596             │ 00_9Z700SML7M │
│ 130001595             │ NULL    

In [8]:
con.sql(
    """
    with merged_couples as (
        SELECT 
            easybill_kundennummer,
            unnest(COALESCE(string_split(medisoft_ids, '\n'), [NULL])) AS medisoft_id,
            'Berlin' as city
        FROM Berlin

        union all

        SELECT 
            easybill_kundennummer,
            unnest(COALESCE(string_split(medisoft_ids, '\n'), [NULL])) AS medisoft_id,
            'Düsseldorf' as city
        FROM Düsseldorf

        union all

        SELECT 
            easybill_kundennummer,
            unnest(COALESCE(string_split(medisoft_ids, '\n'), [NULL])) AS medisoft_id,
            'Frankfurt' as city
        FROM Frankfurt

        union all

        SELECT 
            easybill_kundennummer,
            unnest(COALESCE(string_split(medisoft_ids, '\n'), [NULL])) AS medisoft_id,
            'Hamburg' as city
        FROM Hamburg

        union all

        SELECT 
            easybill_kundennummer,
            unnest(COALESCE(string_split(medisoft_ids, '\n'), [NULL])) AS medisoft_id,
            'Kiel' as city
        FROM Kiel

        union all

        SELECT 
            easybill_kundennummer,
            unnest(COALESCE(string_split(medisoft_ids, '\n'), [NULL])) AS medisoft_id,
            'Köln' as city
        FROM Köln

        union all

        SELECT 
            easybill_kundennummer,
            unnest(COALESCE(string_split(medisoft_ids, '\n'), [NULL])) AS medisoft_id,
            'München' as city
        FROM München

        union all

        SELECT 
            easybill_kundennummer,
            unnest(COALESCE(string_split(medisoft_ids, '\n'), [NULL])) AS medisoft_id,
            'Rostock' as city
        FROM Rostock

        union all

        SELECT 
            easybill_kundennummer,
            unnest(COALESCE(string_split(medisoft_ids, '\n'), [NULL])) AS medisoft_id,
            'Stuttgart' as city
        FROM Stuttgart

        union all

        SELECT 
            easybill_kundennummer,
            unnest(COALESCE(string_split(medisoft_ids, '\n'), [NULL])) AS medisoft_id,
            'Viersen' as city
        FROM Viersen    
    ), distinct_couples as (
        select 
            distinct easybill_kundennummer, medisoft_id
        from merged_couples
    )
    select d.easybill_kundennummer, d.medisoft_id
    from distinct_couples d
    where (d.easybill_kundennummer is not null
        or not exists(
            select 1
            from distinct_couples d2
            where d2.easybill_kundennummer = d.easybill_kundennummer
                and d2.medisoft_id is not null
        ))
    order by d.easybill_kundennummer
    """
).to_csv('output/notebook_extract.csv')

In [10]:
for source in SPREADSHEET_MAPPING.sheets[:]:
    table_name = _sql_identifier("medisoft_" + source.city)
    spreadsheet_id = _sql_literal(source.spreadsheet_id)
    sheet_name = _sql_literal(source.medisoft_spreadsheet_name)

    con.execute(
        f"""
CREATE OR REPLACE TABLE {table_name} AS
SELECT *
FROM read_gsheet(
    '{spreadsheet_id}',
    sheet='{sheet_name}',
    all_varchar=true
);
"""
    )
    print(table_name)

con.sql("SHOW TABLES").df()

"medisoft_Düsseldorf"
"medisoft_Frankfurt"
"medisoft_Hamburg"
"medisoft_Kiel"
"medisoft_Köln"
"medisoft_München"
"medisoft_Rostock"
"medisoft_Stuttgart"
"medisoft_Viersen"
"medisoft_Berlin"


,name
0,Berlin
1,Düsseldorf
2,Frankfurt
3,Hamburg
4,Kiel
5,Köln
6,München
7,Rostock
8,Stuttgart
9,Viersen


In [14]:
queries = [
    f"""
    select *
    from {_sql_identifier("medisoft_" + source.city)}
    """
    for source in SPREADSHEET_MAPPING.sheets
]

con.sql(
    f"""
    create or replace table medisoft_all as 
    {" UNION ALL ".join(queries)}
    """
    )

In [32]:
con.sql("""
    select *
    from medisoft_all 
    where "migrate as inactive" = 'FALSE'
    and "no migration" = 'FALSE'
    and "has easybill connection" = 'FALSE'
    """)

┌──────────────────────────────────────┬───────────────────────────────────────────────────┬───────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────┬─────────────┬────────────────┬──────────────────────────────────────────────────┬─────────────────────────┬────────────────────┬─────────────────────┬──────────────┐
│             medisoft_id              │                       name                        │                      kuerzel                      │                               pfad                                │ nb_patients │ last_exam_date │                     addresse                     │ has easybill connection │ link to connection │ migrate as inactive │ no migration │
│               varchar                │                      varchar                      │                      varchar                      │                              varchar                              │   varchar   │    varchar     

In [17]:
con.sql("select count(distinct medisoft_id) from read_csv('output/easybill_medisoft_pairs.csv')")

┌─────────────────────────────┐
│ count(DISTINCT medisoft_id) │
│            int64            │
├─────────────────────────────┤
│                        2391 │
└─────────────────────────────┘

In [34]:
con.sql(
    """
    select easybill_kundennummer, zoho_id from Berlin
    """
)

┌───────────────────────┬─────────────────────────┐
│ easybill_kundennummer │         zoho_id         │
│        varchar        │         varchar         │
├───────────────────────┼─────────────────────────┤
│ 130002124             │ NULL                    │
│ 130000200             │ zcrm_386758000010045564 │
│ 130000437             │ zcrm_386758000025371007 │
│ 130002066             │ NULL                    │
│ 111000009             │ zcrm_386758000010019168 │
│ 113010031             │ zcrm_386758000010041451 │
│ 130000392             │ zcrm_386758000025186099 │
│ 127000001             │ zcrm_386758000012489188 │
│ 127000000             │ zcrm_386758000010043421 │
│ 130000975             │ zcrm_386758000036443257 │
│     ·                 │  ·                      │
│     ·                 │  ·                      │
│     ·                 │  ·                      │
│ 130002257             │ NULL                    │
│ 130002263             │ NULL                    │
│ 130002223 